# Quickstart — LBO Simulator

This notebook demonstrates a complete LBO simulation from scratch using synthetic company data.

In [ ]:
import sys
sys.path.insert(0, '../src')

from lbo_simulator.data.synthetic import SyntheticCompanyGenerator
from lbo_simulator.models.lbo_engine import LBOEngine
from lbo_simulator.models.covenants import CovenantEngine
from lbo_simulator.models.schemas import (
    DebtTrancheSchema,
    SourcesAndUsesSchema,
    ExitAssumptionsSchema,
    LBOConfigSchema,
    CovenantThresholdsSchema,
)
import pandas as pd

## 1. Generate Synthetic Company

In [ ]:
gen = SyntheticCompanyGenerator(seed=42)
company = gen.get_company_profile("TechCo", sector="SaaS", initial_revenue=100_000_000)

print(f"Company: {company.name}")
print(f"Sector: {company.sector}")
print(f"Revenue: ${company.initial_revenue:,.0f}")
print(f"EBITDA Margin: {company.initial_ebitda_margin:.1%}")
print(f"Revenue Growth: {[f'{g:.0%}' for g in company.revenue_growth_rates]}")

## 2. Build Capital Structure

In [ ]:
# Calculate EBITDA and purchase price
initial_ebitda = company.initial_revenue * company.initial_ebitda_margin
entry_multiple = 8.25
purchase_price = initial_ebitda * entry_multiple

# Capital structure
leverage_pct = 0.55
total_debt = purchase_price * leverage_pct
senior_debt = total_debt * 0.55
mezz_debt = total_debt * 0.25
hy_debt = total_debt * 0.20
equity = purchase_price - total_debt

print(f"Purchase Price: ${purchase_price:,.0f}")
print(f"Total Debt: ${total_debt:,.0f} ({leverage_pct:.0%})")
print(f"Equity: ${equity:,.0f} ({1-leverage_pct:.0%})")

## 3. Define Tranches

In [ ]:
tranches = [
    DebtTrancheSchema(
        name="Senior Term Loan B",
        tranche_type="senior_term_b",
        principal=senior_debt,
        interest_rate=0.075,
        amortization_rate=0.01,
        maturity_years=7.0,
        cash_sweep_rate=0.50,
    ),
    DebtTrancheSchema(
        name="Mezzanine Debt",
        tranche_type="mezzanine",
        principal=mezz_debt,
        interest_rate=0.10,
        amortization_rate=0.0,
        maturity_years=5.0,
        pik_toggle=True,
        pik_rate=0.02,
    ),
    DebtTrancheSchema(
        name="High Yield Bonds",
        tranche_type="high_yield",
        principal=hy_debt,
        interest_rate=0.09,
        amortization_rate=0.0,
        maturity_years=6.0,
    ),
]

for t in tranches:
    print(f"{t.name}: ${t.principal/1e6:.0f}M @ {t.interest_rate:.1%}")

## 4. Run LBO Simulation

In [ ]:
config = LBOConfigSchema(
    company=company,
    sources_and_uses=SourcesAndUsesSchema(
        equity_contribution=equity,
        senior_debt=senior_debt,
        mezzanine_debt=mezz_debt,
        high_yield_debt=hy_debt,
        purchase_price=purchase_price,
        transaction_fees=purchase_price * 0.01,
    ),
    tranches=tranches,
    exit_assumptions=ExitAssumptionsSchema(
        hold_period_years=5,
        exit_ebitda_multiple=9.5,
        entry_ebitda_multiple=entry_multiple,
    ),
    covenants=CovenantThresholdsSchema(
        max_total_leverage=5.5,
        min_fixed_charge_coverage=1.1,
        min_interest_coverage=1.8,
    ),
)

engine = LBOEngine(config)
results = engine.run()

print(f"\n📊 Results:")
print(f"  Equity IRR:  {results.irr:.1%}")
print(f"  MOIC:        {results.moic:.2f}x")
print(f"  Payback:     {results.payback_period_years:.1f} years")
print(f"  Exit EV:     ${results.exit_enterprise_value/1e6:.0f}M")
print(f"  Exit Equity: ${results.exit_equity_value/1e6:.0f}M")

## 5. Cash Flow Waterfall

In [ ]:
cf_df = pd.DataFrame([
    {
        'Year': cf.year,
        'Revenue ($M)': cf.revenue / 1e6,
        'EBITDA ($M)': cf.ebitda / 1e6,
        'Unlevered FCF ($M)': cf.unlevered_fcf / 1e6,
        'Debt Service ($M)': (cf.mandatory_amortization + cf.optional_sweep) / 1e6,
        'Equity Dist ($M)': cf.equity_distribution / 1e6,
    }
    for cf in results.annual_cash_flows
])
cf_df

## 6. Covenant Compliance

In [ ]:
covenant_engine = CovenantEngine(config.covenants)
breaches = covenant_engine.test_covenants(
    results.annual_cash_flows,
    results.debt_schedule,
    results.remaining_debt_at_exit,
)
results.covenant_breaches = breaches

if breaches:
    print(f"⚠️ {len(breaches)} breach(es) found:")
    for b in breaches:
        print(f"  Year {b.year}: {b.covenant_name} = {b.actual_value:.2f} vs {b.threshold_value:.2f} ({b.severity})")
else:
    print("✅ All covenants compliant")

## 7. Next Steps

- Try `examples/custom_tranches.ipynb` for advanced tranche modeling
- Run the Streamlit dashboard: `streamlit run streamlit_app/app.py`
- Optimize capital structure: `lbo-optimize --config config/lbo_config.yaml`